# ECG Data Analysis for 4-Class Arrhythmia Detection
본 ipynb파일에는 `ECGCvdata.csv` 데이터셋을 분석하여 4가지 심전도 상태(NSR, ARR, AF, CHF)를 결정짓는 핵심 파라미터를 도출하고 시각화하는 과정을 담고 있음

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# 데이터 로드
df = pd.read_csv('/content/drive/MyDrive/project/ECGCvdata.csv')

# 클래스 분포 확인
print("=== ECG Signal Class Distribution ===")
display(df['ECG_signal'].value_counts())

## 1. 머신러닝(Random Forest)을 이용한 핵심 특징(Feature) 추출
55개의 파라미터 중 4가지 증상을 분류하는 데 가장 중요한 역할을 한 상위 10개 지표를 추출

In [ ]:
# 불필요한 컬럼 제거 (Index 성격의 RECORD)
X = df.drop(columns=['ECG_signal', 'RECORD'], errors='ignore')
y = df['ECG_signal']

# 결측치(NaN) 중앙값으로 대체 (Random Forest 학습을 위함)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Random Forest 모델 학습
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_imputed, y)

# 변수 중요도(Feature Importance) 추출 및 정렬
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

top_n = 10
top_features = X.columns[indices][:top_n].tolist()

print("Top 10 Important Features:", top_features)

# 변수 중요도 시각화
plt.figure(figsize=(10, 6))
plt.title("Top 10 Feature Importances for 4-Class Classification")
plt.bar(range(top_n), importances[indices][:top_n], align="center", color='skyblue', edgecolor='black')
plt.xticks(range(top_n), top_features, rotation=45)
plt.xlim([-1, top_n])
plt.ylabel('Importance Score')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 증상별(Class) 상위 특징 분포 시각화 (Boxplot)
도출된 상위 특징들이 각 클래스(NSR, ARR, AF, CHF)별로 어떤 차이와 분포를 가지는지 Boxplot으로 시각화

SNN 모델의 Threshold(T_low, T_high 등)를 최적화할 때 이 분포를 참고

In [ ]:
# 상위 8개 특징에 대한 Boxplot 시각화
plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

for i, feature in enumerate(top_features[:8]):
    plt.subplot(2, 4, i+1)

    # (hue 및 legend 옵션 추가)
    sns.boxplot(
        x='ECG_signal',
        y=feature,
        data=df,

        # NSR : 정상 심전도
        # ARR : 부정맥(조기 수축 등)
        # AF : 심방세동
        # CHF : 울혈성 심부전

        order=['NSR', 'ARR', 'AF', 'CHF'],
        hue='ECG_signal',  # 추가됨
        legend=False,      # 추가됨
        palette="Set2"
    )

    plt.title(f'Distribution of {feature}', fontsize=12, fontweight='bold')
    plt.xlabel('')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

## 3. 통계적 평균값 비교
상위 특징들의 클래스별 평균값을 수치로 직접 확인합니다.

In [ ]:
# 상위 10개 변수의 증상별 평균
class_mean = df.groupby('ECG_signal')[top_features].mean().T
class_mean = class_mean[['NSR', 'ARR', 'AF', 'CHF']] # 순서 정렬
display(class_mean.round(2))

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

# 데이터 로드 (경로는 본인의 Colab 환경에 맞게 유지)
df = pd.read_csv('/content/drive/MyDrive/project/ECGCvdata.csv')

# 독립변수(X)와 종속변수(y) 분리
X = df.drop(columns=['ECG_signal', 'RECORD'], errors='ignore')
y = df['ECG_signal']

# 결측치(NaN) 처리 (중앙값 대체)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# 의사결정나무 모델 학습
# 핵심: max_depth=3 으로 설정하여 트리가 너무 복잡해지지 않고 핵심 기준값만 보이도록 제한했습니다.
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_clf.fit(X_imputed, y)

# 트리 구조 시각화
plt.figure(figsize=(20, 10))
plot_tree(
    dt_clf,
    feature_names=X.columns,       # 각 노드에 파라미터 이름 표시
    class_names=dt_clf.classes_,   # 각 노드의 다수결 클래스 (AF, ARR, CHF, NSR) 표시
    filled=True,                   # 클래스별로 색상 채우기
    rounded=True,                  # 노드 모서리를 둥글게
    fontsize=12                    # 글자 크기
)

plt.title("Decision Tree for ECG Classification (Threshold Rules)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()